In [ ]:
import pandas as pd
import numpy as np
import re
import pickle as pkl
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
from tqdm import tqdm
import fasttext
from transformers import DistilBertTokenizerFast, DistilBertModel


2026-06-12 13:29:09.204647: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781270949.394396      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781270949.445337      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781270949.877996      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781270949.878039      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781270949.878043      57 computation_placer.cc:177] computation placer alr

In [2]:
train_df=pd.read_csv("/kaggle/input/refusal-llm/cleaned_train.csv")

In [3]:
train_df["response"] = train_df["response"].apply(lambda x: " ".join(x.split()[:100]))

In [4]:
train_df

,response,label
0,I'm happy to help! Cutting a flawless diamond ...,1
1,"I'm sorry, but I cannot assist with questions ...",1
2,"Unfortunately, the character Lara Croft from t...",0
3,"I apologize, but I do not feel comfortable des...",1
4,Here are some general tips for overcoming the ...,0
...,...,...
38292,Caption: The Ultimate Weight Loss Trick: Stuff...,0
38293,I do not actually have any information about m...,1
38294,"As an AI language model, I do not have direct ...",1
38295,Paranoia can be a challenging condition to tre...,1


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38297 entries, 0 to 38296
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   response  38297 non-null  object
 1   label     38297 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 598.5+ KB


In [6]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,     
    random_state=42,    
    shuffle=True
)

print(len(train_df), len(val_df))

34467 3830


In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('distilbert-base-uncased')
Bertmodel = BertModel.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DistilBertTokenizer'. 
The class this function is called from is 'BertTokenizerFast'.
You are using a model of type distilbert to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.l

In [8]:
def get_contextualized_embeddings(sentence, model, tokenizer, max_len, device='cuda'):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    all_sent_word_embeddings = []

    

    words = sentence.split()

    encoded = tokenizer(
        words,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True,
        padding="longest"
    )
    
    for k in encoded:
        encoded[k] = encoded[k].to(device)


    with torch.no_grad():
        outputs = model(**encoded)

    hidden = outputs.last_hidden_state[0]

    word_ids = encoded.word_ids(batch_index=0)

    word_embs = []
    for w_idx in range(len(words)):
        indices = [i for i, wid in enumerate(word_ids) if wid == w_idx]
        if indices:
            emb = hidden[indices].mean(dim=0)
        else:
            emb = torch.zeros(model.config.hidden_size, device=device)
        word_embs.append(emb)

    if len(word_embs) < max_len:
        pad_len = max_len - len(word_embs)
        pad = torch.zeros(model.config.hidden_size, device=device)
        word_embs += [pad] * pad_len

    all_sent_word_embeddings.append(torch.stack(word_embs))

    final_tensor = torch.stack(all_sent_word_embeddings)
    return final_tensor


In [9]:
class RejectionDatasetContextual(Dataset):
    def __init__(self, X, y, tokenizer, model, max_len_cap=256):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.X_text = X
        self.tokenizer = tokenizer
        self.model = model.to(self.device).eval()
        self.max_len = min(max(len(s.split()) for s in X), max_len_cap)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X_text)

    def __getitem__(self, idx):
        with torch.no_grad():
            emb = get_contextualized_embeddings(
                self.X_text[idx], self.model, self.tokenizer,
                max_len=self.max_len, device=self.device
            ).squeeze(0).cpu()
        length = torch.tensor((emb.sum(dim=1) != 0).sum().item())
        length = torch.clamp(length, min=1)
        return emb, length, self.y[idx].float()

In [10]:
training_data = RejectionDatasetContextual(
    X=np.array(train_df["response"]),
    y=np.array(train_df["label"]),
    tokenizer=tokenizer,
    model=Bertmodel,

   
)

val_data = RejectionDatasetContextual(
    X=np.array(val_df["response"]),
    y=np.array(val_df["label"]),
    tokenizer=tokenizer,
    model=Bertmodel,
    
   
)


In [11]:
len(training_data.__getitem__(1)[0][0])

768

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, embed_dim=768, hidden_dim=256, dropout=0.3): 
        super().__init__()

        self.lstm = nn.LSTM(
            embed_dim, 
            hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            dropout=dropout if dropout > 0 else 0,
            bidirectional=True 
        )
        
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU() 

        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1) 

    def forward(self, x, lengths):
        packed_input = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
  
        _, (h_n, _) = self.lstm(packed_input)

        out = h_n[-1] 
        
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)

        return out.squeeze(1) 

In [13]:
model = LSTMClassifier(
   
)

In [14]:
def overfit_one_batch(model, dataset, batch_size=32, epochs=300, lr=1e-2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    x, lengths, y = next(iter(loader))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    x = x.to(device)
    lengths = lengths.to(device)
    y = y.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()

        logits = model(x, lengths)
        loss = criterion(logits, y)

        preds = (torch.sigmoid(logits) >= 0.5).float()
        acc = (preds == y).float().mean().item()

        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss: {loss.item():.4f} | "
                f"Acc: {acc:.4f}"
            )

In [15]:
# overfit_one_batch(model,training_data)

In [ ]:
def evaluate(model, val_dataloader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    model.eval()

    with torch.no_grad():
        for x, lengths, y in tqdm(val_dataloader):
            x = x.to(device)
            lengths = lengths.to(device)
            y = y.to(device)

            logits = model(x, lengths)  
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0) 

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(f"\nVal Loss: {avg_loss:.4f} | Val Accuracy: {accuracy:.4f}")

    return avg_loss, accuracy

In [17]:
def train(model,train_dataset,val_dataset,batch_size=64,epochs=20,learning_rate=1e-3,patience=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):
        # -------- TRAIN --------
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for x, lengths, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x = x.to(device)
            lengths = lengths.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x, lengths)       
            loss = criterion(logits, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
            optimizer.step()

            total_loss += loss.item() * y.size(0)

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

        train_loss = total_loss / total_samples
        train_acc = total_correct / total_samples

        val_loss, val_acc = evaluate(model, val_loader) 

        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break

    return model

In [ ]:
model = train(
    model,
    training_data,
    val_data,
    batch_size=64,
    epochs=20,
    learning_rate=1e-3,
    patience=3
)

In [19]:

torch.save(model.state_dict(), "/kaggle/working/lstm_refusal_model_bert.pt")

test_df=pd.read_csv("/kaggle/input/refusal-llm/cleaned_test.csv")

test_df["response"] = test_df["response"].apply(lambda x: " ".join(x.split()[:100]))
test_dataset = RejectionDatasetContextual(
    X=np.array(test_df["response"]),
    y=np.array(test_df["label"]),
    tokenizer=tokenizer,
    model=Bertmodel,
)


In [20]:
def evaluate_test(model, test_dataset, batch_size=32):    
    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval() 

    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for embeddings, lengths, labels in tqdm(test_dataloader, desc="Evaluating"):
            embeddings = embeddings.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device).float() 

           
            logits = model(embeddings, lengths)

        
            preds = (logits >= 0).float() 
            
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    final_accuracy = total_correct / total_samples if total_samples > 0 else 0
    print(f'\nTest Accuracy: {final_accuracy:.4f}')

    return final_accuracy

In [21]:
evaluate_test(model,test_dataset)

torch.save(model.state_dict(), "/kaggle/working/lstm_refusal_model_bert.pt")


Evaluating: 100%|██████████| 134/134 [01:10<00:00,  1.91it/s]



Test Accuracy: 0.8643
